## Step 14 — finalization to zones_5, including zones_4a and zones_4b
**# of cells in notebook:** 3

**Purpose:** After step 13, there are still features with populations <100. First we identify low population features with shared edge neighbors in another zone and merge across zones. There may be islands that failed to merge with features that they should have merged with because of changes in concat values after `zones_3` merging (4b). We correct these cases. Or it may be that a zone’s population is <100 and the island(s) in that zone need to be merged with another zone (4a).   

**Input:**

- `zones_4`
  
**Output:** `zones_4a`, `zones_4b`, `zones_5`

**Main logic:**

**Cell 1 — Create zones_4a from zones_4**

1. Copy `zones_4` to a new working/output layer called `zones_4a`. 
2. Convert selected ID fields, especially `zone2_id` and `FID_zones_3_morph_tess_DSLV`, to text so they can store pipe-delimited merge histories such as `12|18|25`. 
3. Identify features with population < 100, starting with the lowest-population features first. 
4. For each low-population feature, look for shared-edge neighbors in a **different zone1**. Same-zone neighbors are not eligible in this cell. 
5. Choose the best eligible neighbor using this priority order: first same admin and same density; then different admin but same density; then different density. Within the first available priority tier, choose the neighbor with the longest shared boundary. 
6. Merge the low-population feature into the selected neighbor by unioning geometry, summing numeric fields such as `population`, `block_count`, and `Join_Count`, and appending tracking fields such as `concat`, `zone1`, `admin`, `density`, and IDs using pipe-delimited histories. 
7. Stop when no remaining low-population feature has an eligible different-zone shared-edge neighbor. The script reports remaining low-population features, including islands and features that only have same-zone neighbors.

**Cell 2 — Create zones_4b from zones_4a**

1. Copy `zones_4a` to a new output layer called `zones_4b`. 
2. Build a neighbor dictionary to identify **island features**, meaning features with no shared-edge neighbors. 
3. Focus only on low-population islands where `population < 100`. 
4. For each low-population island, choose a non-island target feature using “B case” rules. The preferred target is a feature with the same primary `concat` value. 
5. If no same-`concat` target exists, choose a target in the same `zone1` but with different `density`, preferring same `admin` before different `admin`. Within each candidate group, choose the target with the largest population. 
6. Merge the island into the chosen target by unioning geometry, summing numeric fields, and appending tracking fields with pipe-delimited histories. 
7. Write a `zones_4b_merge_log` table that records each island merge, including the source feature, target feature, merge reason, population before/after, and changes to `concat`, `zone1`, `admin`, and `density`.

**Cell 3 — Create zones_5 from zones_4b**

1. Copy `zones_4b` to a new output layer called `zones_5`. 
2. Identify any remaining features with `population < 100`, again processing the lowest-population feature first. 
3. For each remaining low-population feature, find the nearest other feature. By default, distance is measured from the low-population feature’s convex hull to the target feature’s actual geometry. 
4. If multiple targets are effectively tied, the script resolves candidates by nearest distance, then larger target population, then lower `OBJECTID`. 
5. Merge the low-population source feature into the nearest target by unioning geometry, summing numeric fields, and appending pipe-delimited histories for fields like `concat`, `zone1`, `admin`, `density`, `zone2_id`, and `FID_zones_3_morph_tess_DSLV`. 
6. Continue until no features remain below the population threshold, then write a `zones_5_merge_log` table documenting each nearest-feature merge.

In [ ]:
import arcpy
import os
import math

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"

input_fc = os.path.join(zones_gdb, "zones_4")
output_fc = os.path.join(zones_gdb, "zones_4a")

POP_THRESHOLD = 100
MIN_SHARED_LENGTH = 0.0

# Fields used for merge decision
zone_field = "zone1"
admin_field = "admin"
density_field = "density"

# Numeric fields to convert to text so they can store pipe histories
fields_to_convert_to_text = {
    "zone2_id": "zone2id_txt",
    "FID_zones_3_morph_tess_DSLV": "mtessid_txt"
}

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_number(value):
    if value is None:
        return 0
    try:
        if isinstance(value, float) and math.isnan(value):
            return 0
    except Exception:
        pass
    return value


def pipe_tokens(value):
    if value is None:
        return []

    s = str(value).strip()
    if s == "":
        return []

    return [x.strip() for x in s.split("|") if x.strip() != ""]


def primary_token(value):
    """
    For merge-decision comparisons, use the first token.

    Example:
        '28|11|10' is treated as '28'
    """
    tokens = pipe_tokens(value)
    if not tokens:
        return ""
    return tokens[0]


def append_pipe_values(base_value, added_value):
    """
    Appends added_value tokens after base_value tokens while preserving order.

    Example:
        base_value  = '28|11'
        added_value = '10'
        result      = '28|11|10'

    Duplicate tokens are not added again.
    """
    result = []

    for token in pipe_tokens(base_value) + pipe_tokens(added_value):
        if token not in result:
            result.append(token)

    if not result:
        return None

    return "|".join(result)


def require_fields(fc, required_fields):
    existing_fields = [f.name for f in arcpy.ListFields(fc)]
    missing = [f for f in required_fields if f not in existing_fields]

    if missing:
        raise ValueError(
            "Missing required fields:\n"
            + "\n".join(missing)
            + "\n\nAvailable fields are:\n"
            + "\n".join(existing_fields)
        )


def convert_fields_to_text(fc, conversion_map, text_length=500):
    """
    Converts selected fields to text in-place.

    Since ArcGIS cannot directly change an Integer field to String,
    this function:
        1. Adds a temporary text field.
        2. Copies values as strings.
        3. Deletes the original field.
        4. Renames the temporary field back to the original name.
    """
    existing = {f.name: f for f in arcpy.ListFields(fc)}

    for original_field, temp_field in conversion_map.items():
        existing = {f.name: f for f in arcpy.ListFields(fc)}

        if original_field not in existing:
            print(f"WARNING: Field not found; cannot convert to text: {original_field}")
            continue

        original_type = existing[original_field].type

        if original_type == "String":
            print(f"Field is already text; no conversion needed: {original_field}")
            continue

        if temp_field in existing:
            print(f"Deleting existing temporary field: {temp_field}")
            arcpy.management.DeleteField(fc, temp_field)

        print(f"Converting {original_field} from {original_type} to String...")

        arcpy.management.AddField(
            in_table=fc,
            field_name=temp_field,
            field_type="TEXT",
            field_length=text_length
        )

        with arcpy.da.UpdateCursor(fc, [original_field, temp_field]) as cur:
            for row in cur:
                if row[0] is None:
                    row[1] = None
                else:
                    row[1] = str(row[0])
                cur.updateRow(row)

        arcpy.management.DeleteField(fc, original_field)

        arcpy.management.AlterField(
            in_table=fc,
            field=temp_field,
            new_field_name=original_field,
            new_field_alias=original_field
        )

        print(f"Converted field to text: {original_field}")


def build_neighbor_dict(fc, scratch_gdb):
    """
    Builds:
        {src_oid: [(nbr_oid, shared_length), ...]}

    Uses PolygonNeighbors and keeps only neighbors with shared boundary length > 0.
    This avoids point-touch-only neighbors.
    """
    nbr_table = os.path.join(scratch_gdb, "tmp_zones4a_neighbors")

    if arcpy.Exists(nbr_table):
        arcpy.management.Delete(nbr_table)

    arcpy.analysis.PolygonNeighbors(
        in_features=fc,
        out_table=nbr_table,
        in_fields=None,
        area_overlap="NO_AREA_OVERLAP",
        both_sides="BOTH_SIDES"
    )

    nbr_fields = [f.name for f in arcpy.ListFields(nbr_table)]

    if "src_OBJECTID" not in nbr_fields or "nbr_OBJECTID" not in nbr_fields:
        raise ValueError("PolygonNeighbors output is missing src_OBJECTID or nbr_OBJECTID.")

    length_field = "LENGTH" if "LENGTH" in nbr_fields else None

    read_fields = ["src_OBJECTID", "nbr_OBJECTID"]
    if length_field:
        read_fields.append(length_field)

    nbr_dict = {}

    with arcpy.da.SearchCursor(nbr_table, read_fields) as cur:
        for row in cur:
            src_oid = row[0]
            nbr_oid = row[1]

            if src_oid is None or nbr_oid is None:
                continue

            if length_field:
                shared_len = safe_number(row[2])
            else:
                shared_len = 1

            if shared_len > MIN_SHARED_LENGTH:
                nbr_dict.setdefault(src_oid, []).append((nbr_oid, shared_len))

    arcpy.management.Delete(nbr_table)

    return nbr_dict


def get_low_population_oids(fc, oid_field, pop_field):
    lows = []

    with arcpy.da.SearchCursor(fc, [oid_field, pop_field]) as cur:
        for oid, pop in cur:
            pop_val = safe_number(pop)
            if pop_val < POP_THRESHOLD:
                lows.append((oid, pop_val))

    lows.sort(key=lambda x: x[1])
    return lows


def read_attributes_by_oid(fc, oid_field, fields, oid_list):
    if not oid_list:
        return {}

    oid_string = ", ".join(str(x) for x in oid_list)
    where = f"{arcpy.AddFieldDelimiters(fc, oid_field)} IN ({oid_string})"

    out = {}

    with arcpy.da.SearchCursor(fc, fields, where_clause=where) as cur:
        for row in cur:
            row_dict = dict(zip(fields, row))
            out[row_dict[oid_field]] = row_dict

    return out


def choose_best_neighbor(
    target_attrs,
    neighbor_candidates,
    attrs_by_oid,
    zone_field,
    admin_field,
    density_field
):
    """
    Only neighbors in a different zone are eligible.

    Zone/admin/density comparisons use the first pipe token.

    Priority:
        1. same admin, same density
        2. different admin, same density
        3. different density

    Within each tier, choose longest shared boundary.
    """

    target_zone = primary_token(target_attrs[zone_field])
    target_admin = primary_token(target_attrs[admin_field])
    target_density = primary_token(target_attrs[density_field])

    tier_1 = []
    tier_2 = []
    tier_3 = []

    for nbr_oid, shared_len in neighbor_candidates:
        nbr_attrs = attrs_by_oid[nbr_oid]

        nbr_zone = primary_token(nbr_attrs[zone_field])
        nbr_admin = primary_token(nbr_attrs[admin_field])
        nbr_density = primary_token(nbr_attrs[density_field])

        if nbr_zone == target_zone:
            continue

        same_admin = nbr_admin == target_admin
        same_density = nbr_density == target_density

        if same_admin and same_density:
            tier_1.append((nbr_oid, shared_len))
        elif same_density:
            tier_2.append((nbr_oid, shared_len))
        else:
            tier_3.append((nbr_oid, shared_len))

    for tier in [tier_1, tier_2, tier_3]:
        if tier:
            tier.sort(key=lambda x: x[1], reverse=True)
            return tier[0][0], tier[0][1]

    return None, None


# ------------------------------------------------------------
# Prepare output
# ------------------------------------------------------------
if not arcpy.Exists(input_fc):
    raise FileNotFoundError(f"Input feature class not found: {input_fc}")

if arcpy.Exists(output_fc):
    arcpy.management.Delete(output_fc)
    print(f"Deleted existing output: {output_fc}")

print("Copying zones_4 to zones_4a...")
arcpy.management.CopyFeatures(input_fc, output_fc)

oid_field = arcpy.Describe(output_fc).OIDFieldName

# ------------------------------------------------------------
# Convert selected numeric ID fields to text
# ------------------------------------------------------------
convert_fields_to_text(
    fc=output_fc,
    conversion_map=fields_to_convert_to_text,
    text_length=500
)

# ------------------------------------------------------------
# Validate fields
# ------------------------------------------------------------
required_fields = [
    "population",
    "block_count",
    "Join_Count",
    zone_field,
    admin_field,
    density_field,
    "concat",
    "zone2_id",
    "FID_zones_3_morph_tess_DSLV"
]

require_fields(output_fc, required_fields)

existing_fields = [f.name for f in arcpy.ListFields(output_fc)]

# ------------------------------------------------------------
# Define field behavior during merges
# ------------------------------------------------------------

# Numeric fields to sum
sum_fields = [
    "population",
    "block_count",
    "Join_Count"
]

# Text fields to pipe-update
pipe_fields = [
    "concat",
    "zone1",
    "admin",
    "density",
    "zone2_id",
    "FID_zones_3_morph_tess_DSLV"
]

# Confirm pipe fields are text
field_info = {f.name: f for f in arcpy.ListFields(output_fc)}

for fld in pipe_fields:
    if field_info[fld].type != "String":
        raise TypeError(
            f"Field '{fld}' must be String to receive pipe-delimited values, "
            f"but it is {field_info[fld].type}."
        )

print(f"Using population field: population")
print(f"Using zone field:       {zone_field}")
print(f"Using admin field:      {admin_field}")
print(f"Using density field:    {density_field}")

print("\nNumeric fields summed during merges:")
for fld in sum_fields:
    print(f"  - {fld}")

print("\nPipe-updated fields:")
for fld in pipe_fields:
    print(f"  - {fld}")

# ------------------------------------------------------------
# Iterative merging
# ------------------------------------------------------------
scratch_gdb = arcpy.env.scratchGDB
if not scratch_gdb:
    scratch_gdb = zones_gdb

merged_count = 0

print("\nStarting zones_4 to zones_4a merge process...")

while True:
    neighbor_dict = build_neighbor_dict(output_fc, scratch_gdb)
    low_features = get_low_population_oids(output_fc, oid_field, "population")

    merge_candidate = None

    for target_oid, target_pop in low_features:
        neighbor_candidates = neighbor_dict.get(target_oid, [])

        # No physical neighbors: leave for later
        if len(neighbor_candidates) == 0:
            continue

        neighbor_oids = [x[0] for x in neighbor_candidates]
        oids_to_read = [target_oid] + neighbor_oids

        chooser_fields = [
            oid_field,
            zone_field,
            admin_field,
            density_field
        ]

        attrs_by_oid = read_attributes_by_oid(
            output_fc,
            oid_field,
            chooser_fields,
            oids_to_read
        )

        if target_oid not in attrs_by_oid:
            continue

        best_neighbor_oid, best_shared_len = choose_best_neighbor(
            target_attrs=attrs_by_oid[target_oid],
            neighbor_candidates=neighbor_candidates,
            attrs_by_oid=attrs_by_oid,
            zone_field=zone_field,
            admin_field=admin_field,
            density_field=density_field
        )

        # Has neighbors, but none in a different zone: leave for later
        if best_neighbor_oid is None:
            continue

        merge_candidate = (
            target_oid,
            best_neighbor_oid,
            target_pop,
            best_shared_len
        )
        break

    if merge_candidate is None:
        print("No remaining eligible low-population features to merge.")
        break

    target_oid, neighbor_oid, target_pop, shared_len = merge_candidate

    print(
        f"Merging low-pop OBJECTID {target_oid} "
        f"(population={target_pop}) into neighbor OBJECTID {neighbor_oid} "
        f"(shared boundary={shared_len})"
    )

    # --------------------------------------------------------
    # Read full target and neighbor rows
    # --------------------------------------------------------
    read_fields = [
        oid_field,
        "SHAPE@"
    ]

    for fld in sum_fields:
        if fld not in read_fields:
            read_fields.append(fld)

    for fld in pipe_fields:
        if fld not in read_fields:
            read_fields.append(fld)

    rows = read_attributes_by_oid(
        output_fc,
        oid_field,
        read_fields,
        [target_oid, neighbor_oid]
    )

    if target_oid not in rows or neighbor_oid not in rows:
        print("WARNING: Could not read both rows for merge. Stopping.")
        break

    target = rows[target_oid]
    neighbor = rows[neighbor_oid]

    # --------------------------------------------------------
    # Build updated neighbor values
    # --------------------------------------------------------
    updated_values = {}

    updated_values["SHAPE@"] = neighbor["SHAPE@"].union(target["SHAPE@"])

    for fld in sum_fields:
        updated_values[fld] = safe_number(neighbor[fld]) + safe_number(target[fld])

    for fld in pipe_fields:
        updated_values[fld] = append_pipe_values(neighbor[fld], target[fld])

    # --------------------------------------------------------
    # Update neighbor row
    # --------------------------------------------------------
    update_fields = ["SHAPE@"]

    for fld in sum_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    for fld in pipe_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    where_neighbor = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {neighbor_oid}"

    with arcpy.da.UpdateCursor(output_fc, update_fields, where_clause=where_neighbor) as ucur:
        for row in ucur:
            for i, fld in enumerate(update_fields):
                row[i] = updated_values[fld]
            ucur.updateRow(row)

    # --------------------------------------------------------
    # Delete target row
    # --------------------------------------------------------
    where_target = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {target_oid}"

    with arcpy.da.UpdateCursor(output_fc, [oid_field], where_clause=where_target) as dcur:
        for row in dcur:
            dcur.deleteRow()

    merged_count += 1

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
final_count = int(arcpy.management.GetCount(output_fc)[0])
remaining_low = get_low_population_oids(output_fc, oid_field, "population")
neighbor_dict_final = build_neighbor_dict(output_fc, scratch_gdb)

remaining_islands = []
remaining_no_different_zone_neighbor = []
remaining_still_eligible = []

for oid, pop in remaining_low:
    nbrs = neighbor_dict_final.get(oid, [])

    if len(nbrs) == 0:
        remaining_islands.append((oid, pop))
        continue

    neighbor_oids = [x[0] for x in nbrs]
    chooser_fields = [oid_field, zone_field, admin_field, density_field]

    attrs_by_oid = read_attributes_by_oid(
        output_fc,
        oid_field,
        chooser_fields,
        [oid] + neighbor_oids
    )

    if oid not in attrs_by_oid:
        continue

    best_neighbor_oid, best_shared_len = choose_best_neighbor(
        target_attrs=attrs_by_oid[oid],
        neighbor_candidates=nbrs,
        attrs_by_oid=attrs_by_oid,
        zone_field=zone_field,
        admin_field=admin_field,
        density_field=density_field
    )

    if best_neighbor_oid is None:
        remaining_no_different_zone_neighbor.append((oid, pop))
    else:
        remaining_still_eligible.append((oid, pop, best_neighbor_oid))

print("\nDone.")
print(f"Output written to: {output_fc}")
print(f"Total merges performed: {merged_count:,}")
print(f"Final feature count: {final_count:,}")

print("\nRemaining low-population features:")
print(f"  Islands / no physical neighbors:             {len(remaining_islands):,}")
print(f"  Neighbors exist, but none in different zone: {len(remaining_no_different_zone_neighbor):,}")
print(f"  Still eligible under current rules:          {len(remaining_still_eligible):,}")

if remaining_still_eligible:
    print("\nWARNING: Some eligible low-population features remain. Review topology or field values.")

print("\nLow-population islands to handle later:")
for oid, pop in remaining_islands[:50]:
    print(f"  OBJECTID {oid}: population={pop}")

if len(remaining_islands) > 50:
    print(f"  ... plus {len(remaining_islands) - 50:,} more")

print("\nLow-population features with neighbors but no different-zone neighbor:")
for oid, pop in remaining_no_different_zone_neighbor[:50]:
    print(f"  OBJECTID {oid}: population={pop}")

if len(remaining_no_different_zone_neighbor) > 50:
    print(f"  ... plus {len(remaining_no_different_zone_neighbor) - 50:,} more")

In [ ]:
import arcpy
import os
import math

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"

input_fc = os.path.join(zones_gdb, "zones_4a")
output_fc = os.path.join(zones_gdb, "zones_4b")
log_table = os.path.join(zones_gdb, "zones_4b_merge_log")

POP_THRESHOLD = 100
MIN_SHARED_LENGTH = 0.0

zone_field = "zone1"
admin_field = "admin"
density_field = "density"
concat_field = "concat"

# Numeric fields to sum during merges
sum_fields = [
    "population",
    "block_count",
    "Join_Count"
]

# Text/history fields to pipe-update during merges
pipe_fields = [
    "concat",
    "zone1",
    "admin",
    "density",
    "zone2_id",
    "FID_zones_3_morph_tess_DSLV"
]

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_number(value):
    if value is None:
        return 0
    try:
        if isinstance(value, float) and math.isnan(value):
            return 0
    except Exception:
        pass
    return value


def pipe_tokens(value):
    if value is None:
        return []

    s = str(value).strip()
    if s == "":
        return []

    return [x.strip() for x in s.split("|") if x.strip() != ""]


def primary_token(value):
    """
    For comparisons, use the first pipe token.

    Examples:
        20_1_1|2 -> 20_1_1
        20|21    -> 20
    """
    tokens = pipe_tokens(value)
    if not tokens:
        return ""
    return tokens[0]


def append_pipe_values(base_value, added_value):
    """
    Append added_value tokens after base_value tokens while preserving order.
    Duplicate tokens are not added again.

    Example:
        base_value  = 20_1_1|2
        added_value = 20_1_1
        result      = 20_1_1|2
    """
    result = []

    for token in pipe_tokens(base_value) + pipe_tokens(added_value):
        if token not in result:
            result.append(token)

    if not result:
        return None

    return "|".join(result)


def require_fields(fc, required_fields):
    existing_fields = [f.name for f in arcpy.ListFields(fc)]
    missing = [f for f in required_fields if f not in existing_fields]

    if missing:
        raise ValueError(
            "Missing required fields:\n"
            + "\n".join(missing)
            + "\n\nAvailable fields are:\n"
            + "\n".join(existing_fields)
        )


def validate_pipe_fields_are_text(fc, pipe_fields):
    field_info = {f.name: f for f in arcpy.ListFields(fc)}

    for fld in pipe_fields:
        if fld not in field_info:
            raise ValueError(f"Pipe field not found: {fld}")

        if field_info[fld].type != "String":
            raise TypeError(
                f"Field '{fld}' must be String to receive pipe-delimited values, "
                f"but it is {field_info[fld].type}. "
                f"If needed, convert this field to text before running this script."
            )


def build_neighbor_dict(fc, scratch_gdb):
    """
    Builds:
        {src_oid: [(nbr_oid, shared_length), ...]}

    Features absent from the dictionary are treated as islands/no-neighbor features.
    """
    nbr_table = os.path.join(scratch_gdb, "tmp_zones4b_neighbors")

    if arcpy.Exists(nbr_table):
        arcpy.management.Delete(nbr_table)

    arcpy.analysis.PolygonNeighbors(
        in_features=fc,
        out_table=nbr_table,
        in_fields=None,
        area_overlap="NO_AREA_OVERLAP",
        both_sides="BOTH_SIDES"
    )

    nbr_fields = [f.name for f in arcpy.ListFields(nbr_table)]

    if "src_OBJECTID" not in nbr_fields or "nbr_OBJECTID" not in nbr_fields:
        raise ValueError("PolygonNeighbors output is missing src_OBJECTID or nbr_OBJECTID.")

    length_field = "LENGTH" if "LENGTH" in nbr_fields else None

    read_fields = ["src_OBJECTID", "nbr_OBJECTID"]
    if length_field:
        read_fields.append(length_field)

    nbr_dict = {}

    with arcpy.da.SearchCursor(nbr_table, read_fields) as cur:
        for row in cur:
            src_oid = row[0]
            nbr_oid = row[1]

            if src_oid is None or nbr_oid is None:
                continue

            if length_field:
                shared_len = safe_number(row[2])
            else:
                shared_len = 1

            if shared_len > MIN_SHARED_LENGTH:
                nbr_dict.setdefault(src_oid, []).append((nbr_oid, shared_len))

    arcpy.management.Delete(nbr_table)

    return nbr_dict


def read_all_attrs(fc, oid_field, fields):
    out = {}

    with arcpy.da.SearchCursor(fc, fields) as cur:
        for row in cur:
            row_dict = dict(zip(fields, row))
            out[row_dict[oid_field]] = row_dict

    return out


def choose_b_target(source_oid, attrs_by_oid, non_island_oids):
    """
    Chooses a target for a low-population island.

    B.1:
        same primary concat

    B.2:
        if no B.1 target:
            same zone, different density, same admin
            else same zone, different density, different admin

    Within each tier, choose the eligible target with largest population.
    """
    source = attrs_by_oid[source_oid]

    source_primary_concat = primary_token(source[concat_field])
    source_zone = primary_token(source[zone_field])
    source_admin = primary_token(source[admin_field])
    source_density = primary_token(source[density_field])

    b1 = []
    b2_tier1 = []
    b2_tier2 = []

    for target_oid in non_island_oids:
        if target_oid == source_oid:
            continue

        target = attrs_by_oid[target_oid]

        target_primary_concat = primary_token(target[concat_field])
        target_zone = primary_token(target[zone_field])
        target_admin = primary_token(target[admin_field])
        target_density = primary_token(target[density_field])
        target_pop = safe_number(target["population"])

        # B.1: same primary concat
        if target_primary_concat == source_primary_concat:
            b1.append((target_oid, target_pop))
            continue

        # B.2 only applies if same zone, different density
        if target_zone != source_zone:
            continue

        if target_density == source_density:
            continue

        # B.2 tier 1: same zone, different density, same admin
        if target_admin == source_admin:
            b2_tier1.append((target_oid, target_pop))
        # B.2 tier 2: same zone, different density, different admin
        else:
            b2_tier2.append((target_oid, target_pop))

    if b1:
        b1.sort(key=lambda x: x[1], reverse=True)
        return b1[0][0], "B1_same_primary_concat"

    if b2_tier1:
        b2_tier1.sort(key=lambda x: x[1], reverse=True)
        return b2_tier1[0][0], "B2_same_zone_diff_density_same_admin"

    if b2_tier2:
        b2_tier2.sort(key=lambda x: x[1], reverse=True)
        return b2_tier2[0][0], "B2_same_zone_diff_density_diff_admin"

    return None, None


def create_log_table(log_table):
    if arcpy.Exists(log_table):
        arcpy.management.Delete(log_table)
        print(f"Deleted existing log table: {log_table}")

    arcpy.management.CreateTable(
        out_path=os.path.dirname(log_table),
        out_name=os.path.basename(log_table)
    )

    log_fields = [
        ("source_oid", "LONG"),
        ("target_oid", "LONG"),
        ("merge_reason", "TEXT", 100),
        ("source_population", "DOUBLE"),
        ("target_population_before", "DOUBLE"),
        ("target_population_after", "DOUBLE"),
        ("source_concat", "TEXT", 500),
        ("target_concat_before", "TEXT", 500),
        ("target_concat_after", "TEXT", 500),
        ("source_zone1", "TEXT", 500),
        ("target_zone1_before", "TEXT", 500),
        ("target_zone1_after", "TEXT", 500),
        ("source_admin", "TEXT", 500),
        ("target_admin_before", "TEXT", 500),
        ("target_admin_after", "TEXT", 500),
        ("source_density", "TEXT", 500),
        ("target_density_before", "TEXT", 500),
        ("target_density_after", "TEXT", 500),
    ]

    for spec in log_fields:
        if len(spec) == 2:
            name, ftype = spec
            arcpy.management.AddField(log_table, name, ftype)
        else:
            name, ftype, length = spec
            arcpy.management.AddField(log_table, name, ftype, field_length=length)


# ------------------------------------------------------------
# Prepare output
# ------------------------------------------------------------
if not arcpy.Exists(input_fc):
    raise FileNotFoundError(f"Input feature class not found: {input_fc}")

if arcpy.Exists(output_fc):
    arcpy.management.Delete(output_fc)
    print(f"Deleted existing output: {output_fc}")

print("Copying zones_4a to zones_4b...")
arcpy.management.CopyFeatures(input_fc, output_fc)

oid_field = arcpy.Describe(output_fc).OIDFieldName

required_fields = list(set(sum_fields + pipe_fields + [zone_field, admin_field, density_field, concat_field]))
require_fields(output_fc, required_fields)
validate_pipe_fields_are_text(output_fc, pipe_fields)

create_log_table(log_table)

print("Using fields:")
print(f"  Population: {sum_fields[0]}")
print(f"  Zone:       {zone_field}")
print(f"  Admin:      {admin_field}")
print(f"  Density:    {density_field}")
print(f"  Concat:     {concat_field}")

print("\nNumeric fields summed during merges:")
for fld in sum_fields:
    print(f"  - {fld}")

print("\nPipe-updated fields:")
for fld in pipe_fields:
    print(f"  - {fld}")

# ------------------------------------------------------------
# Identify low-population islands and merge B cases
# ------------------------------------------------------------
scratch_gdb = arcpy.env.scratchGDB
if not scratch_gdb:
    scratch_gdb = zones_gdb

print("\nBuilding neighbor dictionary to identify islands...")
neighbor_dict = build_neighbor_dict(output_fc, scratch_gdb)

all_oids = []
with arcpy.da.SearchCursor(output_fc, [oid_field]) as cur:
    for row in cur:
        all_oids.append(row[0])

island_oids = set([oid for oid in all_oids if oid not in neighbor_dict or len(neighbor_dict.get(oid, [])) == 0])
non_island_oids = set([oid for oid in all_oids if oid not in island_oids])

read_fields = [oid_field, "SHAPE@"]

for fld in sum_fields:
    if fld not in read_fields:
        read_fields.append(fld)

for fld in pipe_fields:
    if fld not in read_fields:
        read_fields.append(fld)

attrs_by_oid = read_all_attrs(output_fc, oid_field, read_fields)

low_pop_islands = []

for oid in island_oids:
    pop = safe_number(attrs_by_oid[oid]["population"])
    if pop < POP_THRESHOLD:
        low_pop_islands.append((oid, pop))

low_pop_islands.sort(key=lambda x: x[1])

print(f"Total features:                 {len(all_oids):,}")
print(f"Island features:                {len(island_oids):,}")
print(f"Non-island features:            {len(non_island_oids):,}")
print(f"Low-population island features: {len(low_pop_islands):,}")

merged_count = 0
not_merged = []

log_insert_fields = [
    "source_oid",
    "target_oid",
    "merge_reason",
    "source_population",
    "target_population_before",
    "target_population_after",
    "source_concat",
    "target_concat_before",
    "target_concat_after",
    "source_zone1",
    "target_zone1_before",
    "target_zone1_after",
    "source_admin",
    "target_admin_before",
    "target_admin_after",
    "source_density",
    "target_density_before",
    "target_density_after",
]

# ------------------------------------------------------------
# Merge B cases
# ------------------------------------------------------------
merged_count = 0
not_merged = []
log_rows = []

log_insert_fields = [
    "source_oid",
    "target_oid",
    "merge_reason",
    "source_population",
    "target_population_before",
    "target_population_after",
    "source_concat",
    "target_concat_before",
    "target_concat_after",
    "source_zone1",
    "target_zone1_before",
    "target_zone1_after",
    "source_admin",
    "target_admin_before",
    "target_admin_after",
    "source_density",
    "target_density_before",
    "target_density_after",
]

for source_oid, source_pop in low_pop_islands:

    # The source may already have been removed from memory after a previous merge.
    if source_oid not in attrs_by_oid:
        continue

    target_oid, merge_reason = choose_b_target(
        source_oid=source_oid,
        attrs_by_oid=attrs_by_oid,
        non_island_oids=non_island_oids
    )

    if target_oid is None:
        not_merged.append((source_oid, source_pop, "no_B1_or_B2_target"))
        continue

    source = attrs_by_oid[source_oid]
    target = attrs_by_oid[target_oid]

    print(
        f"Merging island OBJECTID {source_oid} "
        f"(population={source_pop}) into OBJECTID {target_oid} "
        f"reason={merge_reason}"
    )

    target_pop_before = safe_number(target["population"])
    target_pop_after = target_pop_before + safe_number(source["population"])

    source_concat = source[concat_field]
    target_concat_before = target[concat_field]

    source_zone1 = source[zone_field]
    target_zone1_before = target[zone_field]

    source_admin = source[admin_field]
    target_admin_before = target[admin_field]

    source_density = source[density_field]
    target_density_before = target[density_field]

    updated_values = {}

    updated_values["SHAPE@"] = target["SHAPE@"].union(source["SHAPE@"])

    for fld in sum_fields:
        updated_values[fld] = safe_number(target[fld]) + safe_number(source[fld])

    for fld in pipe_fields:
        updated_values[fld] = append_pipe_values(target[fld], source[fld])

    # --------------------------------------------------------
    # Update target row
    # --------------------------------------------------------
    update_fields = ["SHAPE@"]

    for fld in sum_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    for fld in pipe_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    where_target = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {target_oid}"

    with arcpy.da.UpdateCursor(output_fc, update_fields, where_clause=where_target) as ucur:
        for row in ucur:
            for i, fld in enumerate(update_fields):
                row[i] = updated_values[fld]
            ucur.updateRow(row)

    # --------------------------------------------------------
    # Delete source row
    # --------------------------------------------------------
    where_source = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {source_oid}"

    with arcpy.da.UpdateCursor(output_fc, [oid_field], where_clause=where_source) as dcur:
        for row in dcur:
            dcur.deleteRow()

    # --------------------------------------------------------
    # Update in-memory records
    # --------------------------------------------------------
    for fld, val in updated_values.items():
        attrs_by_oid[target_oid][fld] = val

    attrs_by_oid.pop(source_oid, None)
    island_oids.discard(source_oid)

    # --------------------------------------------------------
    # Store log row in memory.
    # Do NOT write to the log table yet.
    # --------------------------------------------------------
    log_rows.append([
        source_oid,
        target_oid,
        merge_reason,
        safe_number(source["population"]),
        target_pop_before,
        target_pop_after,
        source_concat,
        target_concat_before,
        updated_values[concat_field],
        source_zone1,
        target_zone1_before,
        updated_values[zone_field],
        source_admin,
        target_admin_before,
        updated_values[admin_field],
        source_density,
        target_density_before,
        updated_values[density_field],
    ])

    merged_count += 1

# ------------------------------------------------------------
# Write log rows after all feature class edits are complete
# ------------------------------------------------------------
print("\nWriting merge log...")

with arcpy.da.InsertCursor(log_table, log_insert_fields) as log_cur:
    for log_row in log_rows:
        log_cur.insertRow(log_row)

print(f"Log rows written: {len(log_rows):,}")

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
final_count = int(arcpy.management.GetCount(output_fc)[0])

print("\nDone.")
print(f"Output written to: {output_fc}")
print(f"Log table written to: {log_table}")
print(f"Total B merges performed: {merged_count:,}")
print(f"Final feature count: {final_count:,}")

print("\nLow-population islands not merged in this B step:")
print(f"  Count: {len(not_merged):,}")

for oid, pop, reason in not_merged[:50]:
    print(f"  OBJECTID {oid}: population={pop}, reason={reason}")

if len(not_merged) > 50:
    print(f"  ... plus {len(not_merged) - 50:,} more")

In [ ]:
import arcpy
import os
import math

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"

input_fc = os.path.join(zones_gdb, "zones_4b")
output_fc = os.path.join(zones_gdb, "zones_5")
log_table = os.path.join(zones_gdb, "zones_5_merge_log")

POP_THRESHOLD = 100

# Use convex hull of the low-population feature when measuring distance
USE_CONVEX_HULL_FOR_SOURCE = True

# Fields
population_field = "population"

sum_fields = [
    "population",
    "block_count",
    "Join_Count"
]

pipe_fields = [
    "concat",
    "zone1",
    "admin",
    "density",
    "zone2_id",
    "FID_zones_3_morph_tess_DSLV"
]

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_number(value):
    if value is None:
        return 0
    try:
        if isinstance(value, float) and math.isnan(value):
            return 0
    except Exception:
        pass
    return value


def pipe_tokens(value):
    if value is None:
        return []

    s = str(value).strip()
    if s == "":
        return []

    return [x.strip() for x in s.split("|") if x.strip() != ""]


def append_pipe_values(base_value, added_value):
    """
    Appends added_value tokens after base_value tokens while preserving order.
    Duplicate tokens are not added again.

    Example:
        base_value  = 20_1_1|2
        added_value = 5_1_1
        result      = 20_1_1|2|5_1_1
    """
    result = []

    for token in pipe_tokens(base_value) + pipe_tokens(added_value):
        if token not in result:
            result.append(token)

    if not result:
        return None

    return "|".join(result)


def require_fields(fc, required_fields):
    existing_fields = [f.name for f in arcpy.ListFields(fc)]
    missing = [f for f in required_fields if f not in existing_fields]

    if missing:
        raise ValueError(
            "Missing required fields:\n"
            + "\n".join(missing)
            + "\n\nAvailable fields are:\n"
            + "\n".join(existing_fields)
        )


def validate_pipe_fields_are_text(fc, pipe_fields):
    field_info = {f.name: f for f in arcpy.ListFields(fc)}

    for fld in pipe_fields:
        if fld not in field_info:
            raise ValueError(f"Pipe field not found: {fld}")

        if field_info[fld].type != "String":
            raise TypeError(
                f"Field '{fld}' must be String to receive pipe-delimited values, "
                f"but it is {field_info[fld].type}."
            )


def get_read_fields(oid_field):
    read_fields = [
        oid_field,
        "SHAPE@"
    ]

    for fld in sum_fields:
        if fld not in read_fields:
            read_fields.append(fld)

    for fld in pipe_fields:
        if fld not in read_fields:
            read_fields.append(fld)

    return read_fields


def read_all_attrs(fc, oid_field, read_fields):
    attrs = {}

    with arcpy.da.SearchCursor(fc, read_fields) as cur:
        for row in cur:
            row_dict = dict(zip(read_fields, row))
            attrs[row_dict[oid_field]] = row_dict

    return attrs


def get_low_population_oids(attrs_by_oid):
    lows = []

    for oid, attrs in attrs_by_oid.items():
        pop = safe_number(attrs[population_field])
        if pop < POP_THRESHOLD:
            lows.append((oid, pop))

    lows.sort(key=lambda x: x[1])
    return lows


def get_compare_geometry(geom):
    """
    Returns the geometry used for distance comparison.
    """
    if geom is None:
        return None

    if USE_CONVEX_HULL_FOR_SOURCE:
        try:
            return geom.convexHull()
        except Exception:
            return geom

    return geom


def choose_nearest_target(source_oid, attrs_by_oid):
    """
    Finds the nearest other feature to the source feature.

    Distance is measured from the source comparison geometry:
        source convex hull, if USE_CONVEX_HULL_FOR_SOURCE = True
        otherwise source geometry

    Target geometry is the actual target geometry.

    Ties are resolved by:
        1. smaller distance
        2. larger target population
        3. lower OBJECTID
    """
    source = attrs_by_oid[source_oid]
    source_geom = source["SHAPE@"]

    source_compare_geom = get_compare_geometry(source_geom)

    if source_compare_geom is None:
        return None, None

    candidates = []

    for target_oid, target in attrs_by_oid.items():
        if target_oid == source_oid:
            continue

        target_geom = target["SHAPE@"]
        if target_geom is None:
            continue

        try:
            dist = source_compare_geom.distanceTo(target_geom)
        except Exception:
            continue

        target_pop = safe_number(target[population_field])

        candidates.append((target_oid, dist, target_pop))

    if not candidates:
        return None, None

    candidates.sort(key=lambda x: (x[1], -x[2], x[0]))

    best_target_oid, best_dist, best_target_pop = candidates[0]
    return best_target_oid, best_dist


def create_log_table(log_table):
    if arcpy.Exists(log_table):
        arcpy.management.Delete(log_table)
        print(f"Deleted existing log table: {log_table}")

    arcpy.management.CreateTable(
        out_path=os.path.dirname(log_table),
        out_name=os.path.basename(log_table)
    )

    log_fields = [
        ("source_oid", "LONG"),
        ("target_oid", "LONG"),
        ("merge_reason", "TEXT", 100),
        ("distance_to_target", "DOUBLE"),
        ("source_population", "DOUBLE"),
        ("target_population_before", "DOUBLE"),
        ("target_population_after", "DOUBLE"),
        ("source_concat", "TEXT", 1000),
        ("target_concat_before", "TEXT", 1000),
        ("target_concat_after", "TEXT", 1000),
        ("source_zone1", "TEXT", 1000),
        ("target_zone1_before", "TEXT", 1000),
        ("target_zone1_after", "TEXT", 1000),
        ("source_admin", "TEXT", 1000),
        ("target_admin_before", "TEXT", 1000),
        ("target_admin_after", "TEXT", 1000),
        ("source_density", "TEXT", 1000),
        ("target_density_before", "TEXT", 1000),
        ("target_density_after", "TEXT", 1000),
    ]

    for spec in log_fields:
        if len(spec) == 2:
            name, ftype = spec
            arcpy.management.AddField(log_table, name, ftype)
        else:
            name, ftype, length = spec
            arcpy.management.AddField(log_table, name, ftype, field_length=length)


# ------------------------------------------------------------
# Prepare output
# ------------------------------------------------------------
if not arcpy.Exists(input_fc):
    raise FileNotFoundError(f"Input feature class not found: {input_fc}")

if arcpy.Exists(output_fc):
    arcpy.management.Delete(output_fc)
    print(f"Deleted existing output: {output_fc}")

print("Copying zones_4b to zones_5...")
arcpy.management.CopyFeatures(input_fc, output_fc)

oid_field = arcpy.Describe(output_fc).OIDFieldName

required_fields = list(set(sum_fields + pipe_fields))
require_fields(output_fc, required_fields)
validate_pipe_fields_are_text(output_fc, pipe_fields)

create_log_table(log_table)

print("Using input:")
print(f"  {input_fc}")

print("Using output:")
print(f"  {output_fc}")

print("\nNumeric fields summed during merges:")
for fld in sum_fields:
    print(f"  - {fld}")

print("\nPipe-updated fields:")
for fld in pipe_fields:
    print(f"  - {fld}")

print(f"\nUsing convex hull for source distance: {USE_CONVEX_HULL_FOR_SOURCE}")

# ------------------------------------------------------------
# Iterative nearest-feature merging
# ------------------------------------------------------------
read_fields = get_read_fields(oid_field)
attrs_by_oid = read_all_attrs(output_fc, oid_field, read_fields)

log_rows = []
merged_count = 0

print("\nStarting zones_5 nearest-feature merge process...")

while True:
    low_features = get_low_population_oids(attrs_by_oid)

    if not low_features:
        print("No remaining features below population threshold.")
        break

    source_oid, source_pop = low_features[0]

    target_oid, dist_to_target = choose_nearest_target(source_oid, attrs_by_oid)

    if target_oid is None:
        print(
            f"WARNING: Could not find a nearest target for OBJECTID {source_oid}. "
            "Stopping."
        )
        break

    source = attrs_by_oid[source_oid]
    target = attrs_by_oid[target_oid]

    target_pop_before = safe_number(target[population_field])
    target_pop_after = target_pop_before + safe_number(source[population_field])

    print(
        f"Merging OBJECTID {source_oid} "
        f"(population={source_pop}) into nearest OBJECTID {target_oid} "
        f"(distance={dist_to_target}, target pop before={target_pop_before}, "
        f"target pop after={target_pop_after})"
    )

    # Store values for log before update
    source_concat = source["concat"]
    target_concat_before = target["concat"]

    source_zone1 = source["zone1"]
    target_zone1_before = target["zone1"]

    source_admin = source["admin"]
    target_admin_before = target["admin"]

    source_density = source["density"]
    target_density_before = target["density"]

    # Build updated values for target
    updated_values = {}

    updated_values["SHAPE@"] = target["SHAPE@"].union(source["SHAPE@"])

    for fld in sum_fields:
        updated_values[fld] = safe_number(target[fld]) + safe_number(source[fld])

    for fld in pipe_fields:
        updated_values[fld] = append_pipe_values(target[fld], source[fld])

    # Update target row
    update_fields = ["SHAPE@"]

    for fld in sum_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    for fld in pipe_fields:
        if fld not in update_fields:
            update_fields.append(fld)

    where_target = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {target_oid}"

    with arcpy.da.UpdateCursor(output_fc, update_fields, where_clause=where_target) as ucur:
        for row in ucur:
            for i, fld in enumerate(update_fields):
                row[i] = updated_values[fld]
            ucur.updateRow(row)

    # Delete source row
    where_source = f"{arcpy.AddFieldDelimiters(output_fc, oid_field)} = {source_oid}"

    with arcpy.da.UpdateCursor(output_fc, [oid_field], where_clause=where_source) as dcur:
        for row in dcur:
            dcur.deleteRow()

    # Update in-memory target
    for fld, val in updated_values.items():
        attrs_by_oid[target_oid][fld] = val

    # Remove source from memory
    attrs_by_oid.pop(source_oid, None)

    # Log row
    log_rows.append([
        source_oid,
        target_oid,
        "nearest_feature_from_source_convex_hull" if USE_CONVEX_HULL_FOR_SOURCE else "nearest_feature_from_source_geometry",
        dist_to_target,
        safe_number(source[population_field]),
        target_pop_before,
        target_pop_after,
        source_concat,
        target_concat_before,
        updated_values["concat"],
        source_zone1,
        target_zone1_before,
        updated_values["zone1"],
        source_admin,
        target_admin_before,
        updated_values["admin"],
        source_density,
        target_density_before,
        updated_values["density"],
    ])

    merged_count += 1

# ------------------------------------------------------------
# Write log rows after all feature edits are complete
# ------------------------------------------------------------
print("\nWriting merge log...")

log_insert_fields = [
    "source_oid",
    "target_oid",
    "merge_reason",
    "distance_to_target",
    "source_population",
    "target_population_before",
    "target_population_after",
    "source_concat",
    "target_concat_before",
    "target_concat_after",
    "source_zone1",
    "target_zone1_before",
    "target_zone1_after",
    "source_admin",
    "target_admin_before",
    "target_admin_after",
    "source_density",
    "target_density_before",
    "target_density_after",
]

with arcpy.da.InsertCursor(log_table, log_insert_fields) as log_cur:
    for row in log_rows:
        log_cur.insertRow(row)

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
final_count = int(arcpy.management.GetCount(output_fc)[0])

remaining_low = []

with arcpy.da.SearchCursor(output_fc, [oid_field, population_field]) as cur:
    for oid, pop in cur:
        pop_val = safe_number(pop)
        if pop_val < POP_THRESHOLD:
            remaining_low.append((oid, pop_val))

remaining_low.sort(key=lambda x: x[1])

print("\nDone.")
print(f"Output written to: {output_fc}")
print(f"Log table written to: {log_table}")
print(f"Total nearest-feature merges performed: {merged_count:,}")
print(f"Final feature count: {final_count:,}")
print(f"Remaining features below population threshold: {len(remaining_low):,}")

if remaining_low:
    print("\nRemaining low-population features:")
    for oid, pop in remaining_low[:50]:
        print(f"  OBJECTID {oid}: population={pop}")
    if len(remaining_low) > 50:
        print(f"  ... plus {len(remaining_low) - 50:,} more")